# Mondial — ML bake-off (research reproduction)

**What this is.** A zero-setup reproduction of the *exploratory* machine-learning
comparison behind [Mondial](https://cup.shpa.la)'s prediction model — *can classical
ML or neural nets beat the calibrated Elo–Davidson model?* Run it top to bottom
(**Runtime → Run all**).

**What this is _not_.** This notebook does **not** contain the shipped model. Mondial's
production model is **TypeScript** (`lib/prediction.ts`, `lib/montecarlo.ts`,
`lib/model/constants.ts`) and its canonical evaluation is `npm run backtest` plus the
Vitest pins. Re-deriving the shipped Elo/Davidson math in Python is exactly the drift
this notebook avoids — the Elo–Davidson **baseline** below is printed by the committed
`build_features.py`, never a re-implementation.

The narrative write-up and the final verdict live in
[`docs/model-research.md`](https://github.com/shpala/mondial/blob/main/docs/model-research.md).

In [ ]:
# Clone the repo (brings the match corpus + the research scripts) and install the ML deps.
!git clone --depth 1 https://github.com/shpala/mondial.git
%cd mondial/scripts/explore/ml
!pip -q install -r requirements.txt

In [ ]:
# Build the shared, leakage-free feature matrix and print the Elo–Davidson baseline
# (the shipped model's log-loss per split) — the bar every ML model below must beat.
!python build_features.py

## The bar to beat

The `wc2022 ≈ 1.0666` line printed above confirms the Python Elo roll matches the
TypeScript backtest. Every model below trains on the `features.csv` just built and is
scored on the same out-of-sample splits. **Lower log-loss is better;** the broad
`test_general` baseline is ≈ **0.882**.

In [ ]:
# Classical ML over the shared matrix — each script prints its own out-of-sample log-loss.
# (SVM-RBF is slow, and the neural nets need `pip install torch` + the m_nn-*.py scripts;
#  both are omitted here to keep the notebook fast — their results are in model-research.md.)
!python m_logreg-full.py
!python m_random-forest.py
!python m_hist-gbm.py
!python m_xgboost.py

## Result — nothing reliably beats Elo–Davidson out-of-sample

Consolidated from [`docs/model-research.md`](https://github.com/shpala/mondial/blob/main/docs/model-research.md) (broad
`test_general` split, vs the **0.882** baseline):

| Model | OOS log-loss | Verdict |
| --- | --- | --- |
| Logistic regression (all features) | 0.870 | beats on the broad corpus — but the gain is the `games_played` data-reliability feature, meaningless for the uniformly high-cap World Cup field |
| Random forest (calibrated) | 0.876 | marginal |
| XGBoost / HistGBM (calibrated) | 0.90–0.91 | over-confident; lose even after calibration |
| SVM (RBF) | 0.96 | far worse |
| PyTorch MLP / residual / embeddings | ~0.867–0.868 | best raw numbers, but overfit team identities / the gain is a calibration effect, not new signal |

**Conclusion.** None reliably beats the calibrated Elo–Davidson model *on World Cup
matches* — the field is strength-compressed and the sample is small, so extra capacity
fits noise. The one change that *did* help both World Cup holdouts (a World-Cup-specific
probability flattening) shipped; everything here did not. Full reasoning, the leakage-free
evaluation protocol, and the rejected report-card seeds are in
[`docs/model-research.md`](https://github.com/shpala/mondial/blob/main/docs/model-research.md) and
[`docs/algo-bakeoff.md`](https://github.com/shpala/mondial/blob/main/docs/algo-bakeoff.md).